####    기존 분석파일 확인

In [7]:
import os
import multiprocessing
import pandas as pd
from pathlib import Path
from joblib import Parallel, delayed

#   병렬처리 설정
cl = multiprocessing.cpu_count() - 1

#   경로 내 파일 확인
path_001 = r'./'

df_list_001 = list(Path(path_001).rglob('sentiment_*.pkl'))
df_list_001

[WindowsPath('sentiment_tokenized_dataset_260101_260430.pkl'),
 WindowsPath('sentiment_tokenized_dataset_260501_260523.pkl')]

In [8]:
#   해당 파일 불러오기
def read_pkl(f_01):
    return pd.read_pickle(f_01)

df_ls_001 = Parallel(n_jobs = cl)(delayed(read_pkl)(f_01) for f_01 in df_list_001)
df_ls_001

[         섹션                                                 제목       언론사  \
 0       258  잘 나가는 방위산업株…한화에어로·LIG넥스원, 나란히 AA로 신용 ‘레벨업’ [투자...     헤럴드경제   
 1       263                     오세훈 서울시장 “비상계엄 등 잘못 인정하고 반성해야”    이코노미스트   
 2       263         전기차 국고보조금 작년과 동일…내연차 폐차·매각후 전기차 사면 100만원 더      부산일보   
 3       263                    새해 첫날 일부 복권판매점서 '로또발행 일시 중단' 발생      국제신문   
 4       263                  배경훈 "쿠팡, 5개월치 홈피 접속로그 삭제 방치…법 위반"    연합뉴스TV   
 ...     ...                                                ...       ...   
 693916  261                     "우리도 더 달라"...SK하이닉스, 성과급 '후폭풍'    한경비즈니스   
 693917  261                부산상의 "HMM 부산 이전 노사합의 환영…해양수도 부산 성큼"       뉴스1   
 693920  261      SSG닷컴, 6월 '쓱7클럽' VIP 전용 프로모션 신설…“매월 다른 혜택 제공”      전자신문   
 693923  261                        HMM 본사 부산 이전 최종 확정…노사 전격 합의    아이뉴스24   
 693925  771       [르포]첫 선 '대동 AI트랙터' 카메라 6개가 눈, 운전자 없이 동시작업 척척  동행미디어 시대   
 
                                                        본문 Target_Date  an

In [42]:
#   list형태 파일 병합
df_001 = pd.concat(df_ls_001, axis= 0)

df_001.info()

<class 'pandas.DataFrame'>
Index: 614043 entries, 0 to 116114
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   섹션           614043 non-null  int64         
 1   제목           614043 non-null  object        
 2   언론사          614043 non-null  object        
 3   본문           614043 non-null  object        
 4   Target_Date  608806 non-null  datetime64[ns]
 5   answer       608806 non-null  object        
 6   수정           614043 non-null  object        
 7   tokens       614043 non-null  object        
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 42.2+ MB


In [ ]:
#   추가 작업 위해 'answer' 제거
df_002 = df_002.drop(['answer'], axis =1)

In [49]:
#   파일 분리
def divide_by_date(df_01, start_date, end_date, col_01 = 'Target_Date'):
    #   날짜 범위별 파일 생성
    df_02 = df_01[(df_01[col_01] >= start_date) & (df_01[col_01] < end_date)]

    #   범위별 파일명 지정
    df_02[col_01] = pd.to_datetime(df_02[col_01])

    min_date = df_02[col_01].min().strftime('%y%m%d')
    max_date = df_02[col_01].max().strftime('%y%m%d')

    #   경로 지정 및 파일 저장
    file_01 = os.path.join(path_001, 'data', f'df_tokenized_{min_date}_{max_date}.pkl')
    df_02.to_pickle(file_01)

    return df_02


In [ ]:
#   파일 분리 및 파일 저장
df_until_MAR  = divide_by_date(df_002, '2026-01-01', '2026-04-01')
df_until_JUNE  = divide_by_date(df_002, '2026-04-01', '2026-07-01')

In [54]:
#   파일 생성 확인
list(Path(os.path.join(path_001, 'data')).rglob('df_tokenized_*.pkl'))

[WindowsPath('data/df_tokenized_260102_260331.pkl'),
 WindowsPath('data/df_tokenized_260401_260521.pkl')]